# FM study: Colab setup

Puts the frozen-encoder study's files on a fresh Colab runtime and proves they are the right ones.
It runs **no encoder and produces no scores**. The rules are in `docs/research/FM_RUNBOOK.md`.

**Step-by-step guide for whoever runs this:** `docs/research/FM_FIRST_RUN_GUIDE.md`.

**Runtime:** Runtime → Change runtime type → **GPU: A100, L4 or T4**, not RTX PRO 6000 / Blackwell.
**Before you start:** `HF_TOKEN` (a Hugging Face account with RETFound access) must be in Colab Secrets
(key icon) with notebook access on, and your My Drive must have a shortcut to Lam's `octfsl` folder
(guide, Part 1). Lam does not need to run it.

| Step | What it does | What proves it worked |
|---|---|---|
| 1 | Checks the first cell and the GPU | First cell complete; a supported GPU is attached |
| 2 | Clones the exact code and builds a separate Python environment | Commit matches; `pip check` passes; PyTorch sees the GPU |
| 3 | Checks Drive, unpacks the images from `octfsl/raw`, then disconnects Drive | `fm_v1` is writable; all 84,630 manifest images exist |
| 4 | Rebuilds the groups, episodes and rosters | Six files match the laptop's byte for byte |
| 5 | Downloads RETFound, DINOv2, MIRAGE and the Inception reference | Every file matches its pinned SHA256 |
| 6 | Prints the final inventory | Saved for Step 11 |
| 7 | Records each model's identity and Lam's terms decision (pre-filled in the next cell) | One spec file per model |
| 8 | Keeps Drive and credentials away from the model code | Drive is off and no login is cached |
| 9 | Loads each model and times it on 100 images, one process at a time | Every process exits cleanly |
| 10 | Prints the pilot report | Saved for Step 11 |
| 11 | Saves the evidence to `octfsl/fm_v1/first-checks/`, then disconnects Drive | The Drive copy's SHA256 matches |

**Fill in `RUN_BY` and `RETFOUND_ACCESS` in the next cell, then Runtime → Run all.** Any failed check stops
the notebook; Step 1 stops on purpose while the first cell is incomplete or still holds a placeholder. Files live in `/content` and vanish when the runtime ends.
Rerunning is safe.

Steps 7 to 10 use **no labels and produce no scores**. They are first checks, not the final pilots, and
nothing here is the input lock. The evidence for each decision is in `docs/research/FM_TERMS_EVIDENCE.md`.

In [ ]:
# Fill in this cell before Run all (Step 1 stops until it is complete)
# DECISIONS: Lam's terms decision for each model, as Lam finalized it on 2026-11-10 (recorded in
# docs/research/FM_TERMS_EVIDENCE.md, section 4). Do not edit: only Lam changes these.
DECISIONS = {
    "RETFound": "Lam, 2026-11-10. I approve the pinned RETFound_mae_natureOCT weights and models_vit.py as a "
                "frozen encoder for feature extraction and timing in our non-commercial student research, "
                "run by me or a teammate. Licence: CC BY-NC 4.0 for the code and, per the model card I read "
                "on 2026-11-10, for the weights. Each runner uses their own approved Hugging Face account. "
                "Tokens and weights are never shared, saved to Drive, redistributed or committed. "
                "We will cite Zhou et al. 2023 (Nature).",
    "DINOv2": "Lam, 2026-11-10. I approve the pinned facebook/dinov2-base as a frozen encoder for feature "
              "extraction and timing in our non-commercial student research, run by me or a teammate. "
              "Licence: Apache-2.0. The weights are not redistributed or committed. We will cite "
              "Oquab et al. 2024 (TMLR).",
    "MIRAGE": "Lam, 2026-11-10. I approve the pinned MIRAGE-Base weights and mirage_hf.py as a frozen, OCT-only "
              "encoder for feature extraction and timing in our non-commercial student research, run by me "
              "or a teammate. Licence: CC BY-NC 4.0 (non-commercial academic research with attribution). "
              "Weights and code are not redistributed or committed. We will cite Morano et al. 2025 "
              "(npj Digital Medicine).",
    "Inception": "Lam, 2026-11-10. I approve the pinned Keras ImageNet no-top Inception-v3 weights as the "
                 "reference arm only, never nominated, for feature extraction and timing in our "
                 "non-commercial student research, run by me or a teammate. Licence: Apache-2.0 for the "
                 "Keras code and, per the Keras 2.2.4 documentation, for these weights. The weights are "
                 "not redistributed or committed.",
}
# Your name (the person running this notebook).
RUN_BY = ""
# Whose Hugging Face account the HF_TOKEN belongs to, and the date its RETFound access was approved.
RETFOUND_ACCESS = ""

In [ ]:
# Settings and helpers
import csv, datetime, hashlib, json, os, pathlib, shlex, shutil, subprocess, sys, time, zipfile

COMMIT = "41782db6cf3de39155d21f77c0b88971e6317ae2"  # the code being checked; change only on purpose
REPO = "https://github.com/Malnoad/Few-shot-rare-diseases"
ROOT, ENV = "/content/Few-shot", "/content/fm-env"
PY = f"{ENV}/bin/python"
PREPARED, ASSETS = "/content/prepared", "/content/assets"
WORK = "/content/fm_v1"  # everything Step 11 saves: specs, ledger, pilot receipts, logs and reports
LEDGER, REPORTS = f"{WORK}/download_ledger.csv", f"{WORK}/reports"
RAW = "/content/drive/MyDrive/octfsl/raw"  # Lam's folder, through the runner's octfsl shortcut (guide, Part 1)
OUT = "/content/drive/MyDrive/octfsl/fm_v1"  # Lam's output folder, through the same shortcut
IMAGES = 84630
ARMS = ("RETFound", "DINOv2", "MIRAGE", "Inception")  # three FMs and the reference; BioMedCLIP is fallback only

# Produced on the laptop from the same code and manifest (9 Nov).
EXPECTED_PREPARED = {
    "groups.csv": "6991d8229149aadf4af5ee53f1e2eb3991805d3bf56ebb5b367337a30224c5d0",
    "nine_way_roster.csv": "72f90c00b98810af46d257fe53d047d93f5312552ebaa478aef72d58da81b586",
    "episodes_development.csv": "cd060bddb434b5de476f4fd3550b8b21481ebaf3eb597298f727517968544695",
    "episodes_reporting.csv": "f9e1e3c6a4f2aacd405171837d472190271da2b03dbce79b619a03900b202b8f",
    "timing_inventory.csv": "a6be7a5ec3ba1bbe220674faaef8e51105bc7e4da59e9672d20e4e4562be97ed",
    "extraction_inventory.csv": "5ca665e4828b2d45c7d90cc6363708d9548793de7019d417e57222ee2b76b44d",
}


def sh(command):
    """Run a shell command, streaming its output; any failure stops the notebook."""
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"FAILED: {command}")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def report(name, lines):
    """Print a report and keep a copy in REPORTS, so Step 11 can save it to Drive."""
    text = "\n".join(lines)
    print(text)
    path = pathlib.Path(REPORTS, name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text + "\n", encoding="utf-8")

In [ ]:
# Step 1. Check the first cell and the GPU
first_cell = {**DECISIONS, "RUN_BY": RUN_BY, "RETFOUND_ACCESS": RETFOUND_ACCESS}
blank = [name for name, value in first_cell.items() if not value.strip()]
assert not blank, f"Stopped on purpose: fill in {blank} in the first cell, run that cell, then Run all again."
template = [name for name, value in first_cell.items() if "[" in value or "<" in value]
assert not template, f"Stopped on purpose: {template} still contain a [ or < placeholder; replace it with the real text."
query = "nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader"
gpu = subprocess.run(query, shell=True, capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU: Runtime > Change runtime type > GPU (A100, L4 or T4)"
print(gpu.stdout.strip())
assert float(gpu.stdout.split(",")[1]) < 12, "Blackwell GPU: switch to A100, L4 or T4"

In [ ]:
# Step 2. Get the exact code and build the study environment
if not os.path.isdir(ROOT):
    sh(f"git clone -q {REPO} {ROOT}")
sh(f"git -C {ROOT} checkout -q {COMMIT}")
head = subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip()
assert head == COMMIT, head
print("code:", head)

# Separate from Colab's own Python, so its packages cannot conflict. --without-pip needs no ensurepip.
if not os.path.exists(PY):
    sh(f"python3 -m venv --without-pip {ENV}")
sh(f"python3 -m pip --python {PY} install -q -r {ROOT}/requirements-fm.txt")
sh(f"python3 -m pip --python {PY} check")
sh(f"""{PY} -c 'import sys, torch, numpy, scipy, PIL, transformers; assert torch.cuda.is_available(), "PyTorch cannot see the GPU"; print("python", sys.version.split()[0], "torch", torch.__version__, "cuda", torch.version.cuda, "numpy", numpy.__version__)'""")

In [ ]:
# Step 3. Check Drive, unpack the images, then disconnect Drive
from google.colab import drive


def drive_ready():
    """Check both folders now, so a Drive problem cannot surface only at Step 11."""
    for path in (RAW, OUT):
        assert os.path.isdir(path), f"{path} not found: add the octfsl shortcut to your My Drive (guide, Part 1)"
    probe = pathlib.Path(OUT, f".write-test-{time.time_ns()}")
    probe.write_text("ok", encoding="utf-8")
    assert probe.read_text(encoding="utf-8") == "ok", "octfsl/fm_v1 is not writable: ask Lam to check Editor access"
    probe.unlink()
    print("Drive ok: octfsl/raw is readable and octfsl/fm_v1 is writable")


def extract(zip_path, anchor):
    """Copy every file below the folder named `anchor` into data/raw, skipping macOS junk."""
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            parts = [p for p in info.filename.replace("\\", "/").split("/") if p]
            if not parts or info.filename.endswith(("/", "\\")) or "__MACOSX" in parts or parts[-1].startswith("._"):
                continue
            if anchor not in parts[:-1]:
                continue
            target = pathlib.Path(ROOT, "data/raw", *parts[parts.index(anchor):])
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
            count += 1
    print(f"{pathlib.Path(zip_path).name}: {count} files")


with open(f"{ROOT}/data/manifest.csv", newline="", encoding="utf-8") as stream:
    MANIFEST = [row["path"].replace("\\", "/") for row in csv.DictReader(stream)]


def absent():
    return [p for p in MANIFEST if not os.path.isfile(f"{ROOT}/{p}")]


assert len(MANIFEST) == IMAGES, len(MANIFEST)
if absent() or not globals().get("DRIVE_OK"):
    drive.mount("/content/drive", force_remount=True)
    drive_ready()
    if absent():
        extract(f"{RAW}/oct2017.zip", "OCT2017")
        extract(f"{RAW}/rare.zip", "rare")
    drive.flush_and_unmount()
    DRIVE_OK = True
missing_images = absent()
assert not missing_images, f"{len(missing_images)} manifest images missing; first: {missing_images[0]}"
print(len(MANIFEST), "manifest images present")

In [ ]:
# Step 4. Rebuild the rosters and compare them with the laptop's
sh(f"{PY} {ROOT}/fm_prepare.py screen --root {ROOT} --out {PREPARED}")
sh(f"{PY} {ROOT}/fm_prepare.py freeze --root {ROOT} --out {PREPARED}")
different = [name for name, digest in EXPECTED_PREPARED.items() if sha256(f"{PREPARED}/{name}") != digest]
assert not different, f"Differs from the laptop, stop and report: {different}"
print("All", len(EXPECTED_PREPARED), "files match the laptop byte for byte, so the 5,146 images and rosters agree.")

In [ ]:
# Step 5a. Define the checkpoint downloads (revisions and digests come from fm_extract.py)
import requests

pins = json.loads(subprocess.check_output([PY, "-c", (
    "import json, fm_extract as x; print(json.dumps(dict(repositories=x.REPOSITORIES, revisions=x.REVISIONS, "
    "sources=x.SOURCE_REVISIONS, digests=x.EXPECTED_ASSET_SHA256)))")], cwd=ROOT, text=True))


def hf(arm, name):
    return f"https://huggingface.co/{pins['repositories'][arm]}/resolve/{pins['revisions'][arm]}/{name}"


def github(repo, arm, path):
    return f"https://raw.githubusercontent.com/{repo}/{pins['sources'][arm]}/{path}"


KERAS = "inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5"
FILES = [  # arm, file name, source, needs the Hugging Face token
    ("RETFound", "RETFound_mae_natureOCT.pth", hf("RETFound", "RETFound_mae_natureOCT.pth"), True),
    ("RETFound", "models_vit.py", github("rmaphoh/RETFound_MAE", "RETFound", "models_vit.py"), False),
    ("DINOv2", "model.safetensors", hf("DINOv2", "model.safetensors"), False),
    ("DINOv2", "config.json", hf("DINOv2", "config.json"), False),
    ("DINOv2", "preprocessor_config.json", hf("DINOv2", "preprocessor_config.json"), False),
    ("MIRAGE", "model.safetensors", hf("MIRAGE", "model.safetensors"), False),
    ("MIRAGE", "config.json", hf("MIRAGE", "config.json"), False),
    ("MIRAGE", "mirage_hf.py", github("j-morano/MIRAGE", "MIRAGE", "hf/mirage_hf.py"), False),
    ("Inception", KERAS, f"https://storage.googleapis.com/tensorflow/keras-applications/inception_v3/{KERAS}", False),
]
assert {(arm, name) for arm, name, _, _ in FILES} == {(arm, name) for arm in ARMS for name in pins["digests"][arm]}


def record(row):
    """Append to the download-cost ledger the protocol asks for (runbook, R4)."""
    path = pathlib.Path(LEDGER)
    path.parent.mkdir(parents=True, exist_ok=True)
    new = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        if new:
            writer.writerow(["arm", "file", "url", "bytes", "seconds", "utc"])
        writer.writerow(row)


def fetch(arm, name, url, token=None):
    """Download one pinned file. Bytes that fail the digest are deleted, never kept."""
    target = pathlib.Path(ASSETS, arm, name)
    expected = pins["digests"][arm][name]
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and sha256(target) == expected:
        print(f"already verified  {arm}/{name}")
        return
    partial = target.with_name(name + ".part")
    digest, size, start = hashlib.sha256(), 0, time.time()
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    with requests.get(url, headers=headers, stream=True, timeout=60) as response:
        response.raise_for_status()
        with partial.open("wb") as stream:
            for chunk in response.iter_content(1 << 20):
                stream.write(chunk)
                digest.update(chunk)
                size += len(chunk)
    seconds = round(time.time() - start, 1)
    if digest.hexdigest() != expected:
        partial.unlink()
        raise ValueError(f"{arm}/{name} does not match its pinned SHA256; nothing was kept")
    partial.replace(target)
    record([arm, name, url, size, seconds, datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")])
    print(f"downloaded {size / 1e6:9.1f} MB in {seconds:7.1f} s  {arm}/{name}")

In [ ]:
# Step 5b. Download. The token is read only for the RETFound weights and is never stored or printed.
from google.colab import userdata

for arm, name, url, gated in FILES:
    fetch(arm, name, url, userdata.get("HF_TOKEN") if gated else None)

In [ ]:
# Step 6. Final inventory (Step 11 saves it to Drive as step6_inventory.txt)
lines = []
for arm in ARMS:
    for name, expected in pins["digests"][arm].items():
        path = pathlib.Path(ASSETS, arm, name)
        assert sha256(path) == expected, f"{arm}/{name}"
        lines.append(f"OK {path.stat().st_size / 1e6:9.1f} MB  {arm}/{name}")
lines += ["", open(LEDGER, encoding="utf-8").read() if os.path.exists(LEDGER) else "No new downloads this session."]
lines.append("code: " + subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip())
lines.append("Setup complete. No encoder has been run and no score exists.")
report("step6_inventory.txt", lines)

In [ ]:
# Step 7. Record each model's identity and terms (hashes the staged files; downloads nothing)
TERMS = {
    "RETFound": f"Weights: huggingface.co/YukunZhou/RETFound_mae_natureOCT at the pinned revision; gated, downloaded with: {RETFOUND_ACCESS.strip()}. Code: github.com/rmaphoh/RETFound_MAE at the pinned commit, CC BY-NC 4.0.",
    "DINOv2": "huggingface.co/facebook/dinov2-base at the pinned revision, Apache-2.0.",
    "MIRAGE": "huggingface.co/j-morano/MIRAGE-Base at the pinned revision and github.com/j-morano/MIRAGE at the pinned commit, CC BY-NC 4.0 (non-commercial academic research with attribution).",
    "Inception": "Keras ImageNet no-top weights from storage.googleapis.com/tensorflow/keras-applications (no-top MD5 bcbd6486424b2319ff4ef7d526e38f63, as in keras 2.0.6 keras/applications/inception_v3.py). Keras code is Apache-2.0; keras 2.2.4 docs/templates/applications.md (InceptionV3, License) states the weights are released under the Apache License.",
}
CHECKED = "Public pages checked by the agent on 2026-11-10."

undecided = [arm for arm in ARMS if not DECISIONS[arm].strip()]
assert not undecided, f"Stopped on purpose: write your terms decision for {undecided} in the cell at the top, run that cell, then run from here."
SPECS = "/content/fm_v1/specs"
os.makedirs(SPECS, exist_ok=True)
for arm in ARMS:
    evidence = f"{TERMS[arm]} {CHECKED} Lam's decision: {DECISIONS[arm].strip()}"
    source = f"--source-revision {pins['sources'][arm]} " if arm in pins["sources"] else ""
    sh(f"{PY} {ROOT}/fm_extract.py spec --arm {arm} --assets {ASSETS}/{arm} --revision {pins['revisions'][arm]} "
       f"{source}--terms-evidence {shlex.quote(evidence)} --out {SPECS}/{arm}.json")
    print("spec written:", arm)

In [ ]:
# Step 8. Keep Drive and credentials away from the model code (risk record R1)
TOKEN_CACHES = ("/root/.cache/huggingface/token", "/root/.cache/huggingface/stored_tokens", "/root/.huggingface/token")
assert not os.path.ismount("/content/drive"), "Drive is mounted: run drive.flush_and_unmount() first."
cached = [path for path in TOKEN_CACHES if os.path.exists(path)]
assert not cached, f"A Hugging Face login is cached at {cached}. Use a fresh runtime, so the model code cannot read it."
SAFE_ENV = {name: value for name, value in os.environ.items()
            if not any(word in name.upper() for word in ("TOKEN", "SECRET", "KEY", "PASSW", "CREDENTIAL", "HF_", "HUGGING"))}
print("Removed from the worker environment (names only):", sorted(set(os.environ) - set(SAFE_ENV)) or "nothing")

In [ ]:
# Step 9. Load each model and time it on 100 images, one process at a time (no labels, no scores)
ATTEMPT = datetime.datetime.now(datetime.timezone.utc).strftime("first-check-%Y%m%dT%H%M%SZ")
PILOTS = f"/content/fm_v1/pilots/{ATTEMPT}"
INTERPRETER = {**{arm: PY for arm in ARMS}, "Inception": sys.executable}  # the reference uses Colab's own TensorFlow


def worker(command, log):
    """Run one process with no credentials in its environment, keep its output in `log`, wait until it exits."""
    process = subprocess.Popen(command, env=SAFE_ENV, cwd="/content", stdin=subprocess.DEVNULL,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(log, "w", encoding="utf-8") as stream:
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
    return process.pid, process.wait()


def gpu_users():
    return subprocess.check_output(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True).strip()


os.makedirs(PILOTS, exist_ok=True)
RESULTS = {}
for arm in ARMS:  # fixed protocol order; one failed model does not hide the others
    assert not gpu_users(), f"Another process is using the GPU ({gpu_users()}): restart the runtime so timings stay comparable."
    print(f"\n=== {arm} ===", flush=True)
    started = time.time()
    pid, code = worker([INTERPRETER[arm], f"{ROOT}/fm_extract.py", "pilot", "--root", ROOT, "--prepared", PREPARED,
                        "--spec", f"{SPECS}/{arm}.json", "--assets", f"{ASSETS}/{arm}", "--out", f"{PILOTS}/{arm}",
                        "--timing-session", ATTEMPT], f"{PILOTS}/{arm}.log")
    RESULTS[arm] = dict(pid=pid, exit_code=code, seconds=round(time.time() - started, 1))
    print(f"{arm}: exit code {code} after {RESULTS[arm]['seconds']} s", flush=True)
pathlib.Path(PILOTS, "processes.json").write_text(json.dumps(RESULTS, indent=2), encoding="utf-8")

In [ ]:
# Step 10. Pilot report (Step 11 saves it to Drive as step10_pilots.txt; a failure is raised there)
failed, lines = [], []
for arm in ARMS:
    folder = pathlib.Path(PILOTS, arm)
    if RESULTS[arm]["exit_code"] or not (folder / "complete.json").exists():
        failed.append(arm)
        lines.append(f"{arm:10s} FAILED, exit code {RESULTS[arm]['exit_code']}: the error is in the Step 9 output and {arm}.log")
        continue
    timing = json.loads((folder / "timing.json").read_text(encoding="utf-8"))
    load = json.loads((folder / "load_report.json").read_text(encoding="utf-8"))
    costs = timing["costs"]
    unexpected = load.get("unexpected_keys", [])
    notes = {key: load[key] for key in ("checkpoint_layout", "default_norm", "text_tower") if key in load}
    lines.append(f"{arm:10s} {timing['median_seconds_per_image'] * 1000:8.2f} ms/image   load {costs['model_loading_seconds']:6.1f} s   "
                 f"peak GPU memory {costs['peak_vram_bytes'] / 1e9:5.2f} GB")
    lines.append(f"           missing keys: {load.get('missing_keys', [])}   unexpected keys: {len(unexpected)} {unexpected[:3]}   {notes}")
lines += ["", f"attempt: {ATTEMPT}", "gpu: " + gpu.stdout.strip(),
          "code: " + subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip(),
          "No labels were used and no score exists. These are first checks, not final pilots, and not the input lock."]
report("step10_pilots.txt", lines)

In [ ]:
# Step 11. Save this attempt's evidence to octfsl/fm_v1/first-checks, then disconnect Drive
# Every model process has exited. Saved: reports, specs, download ledger, pilot receipts and logs; no weights or images.
evidence = shutil.make_archive(f"/content/{ATTEMPT}", "zip", WORK)
digest = sha256(evidence)
print("local evidence zip:", evidence, digest)
receipt = dict(attempt=ATTEMPT, run_by=RUN_BY.strip(), code=COMMIT, gpu=gpu.stdout.strip(), failed=failed,
               zip=pathlib.Path(evidence).name, zip_bytes=os.path.getsize(evidence), zip_sha256=digest,
               saved_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"))
drive.mount("/content/drive", force_remount=True)
target = pathlib.Path(OUT, "first-checks", ATTEMPT)
assert not target.exists(), f"{target} already exists; nothing was overwritten"
target.mkdir(parents=True)
shutil.copyfile(evidence, target / receipt["zip"])
for name in ("step6_inventory.txt", "step10_pilots.txt"):
    shutil.copyfile(f"{REPORTS}/{name}", target / name)
(target / "receipt.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
copied = sha256(target / receipt["zip"])
drive.flush_and_unmount()
assert copied == digest, f"The Drive copy differs from {evidence}: download that file from the Files panel and send it to Lam."
print(f"Saved to octfsl/fm_v1/first-checks/{ATTEMPT}/  (zip SHA256 {digest})")
if failed:
    raise RuntimeError(f"Pilot failed for {failed}. The evidence is saved; tell Lam. Do not change anything in the notebook.")
print("All four models loaded and timed. Tell Lam the attempt name above.")